# Exploratory Data Analysis — Biomedical Conversational AI

EDA of PubMed abstracts and BioASQ QA datasets for a Conversational AI system with NER-based intent classification.

**Datasets**: PubMed Abstracts (13,200 abstracts, 8 topics) | BioASQ QA (1,997 questions)

**Goals**: NER for biomedical entities, intent classification, conversational QA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import re
import ast
import warnings
from collections import Counter
from pathlib import Path

from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import ngrams

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

print('✅ All libraries loaded successfully')

---
## 1. Data Loading

In [ ]:
# Load PubMed abstracts
raw_df = pd.read_csv('../data/raw/pubmed_abstracts.csv')
print(f'PubMed Abstracts shape: {raw_df.shape}')
print(f'\nColumns: {raw_df.columns.tolist()}')
print(f'\nDtypes:')
print(raw_df.dtypes)
raw_df.head(3)

In [ ]:
def parse_pubmed_entry(entry_str):
    """Parse the string tuple format into (abstract, title)."""
    if pd.isna(entry_str):
        return None, None
    try:
        parsed = ast.literal_eval(entry_str)
        if isinstance(parsed, tuple) and len(parsed) == 2:
            abstract_list, title = parsed
            if isinstance(abstract_list, list):
                abstract = ' '.join(abstract_list)
            else:
                abstract = str(abstract_list)
            return abstract, title
    except (ValueError, SyntaxError):
        pass
    return None, None

# Topic columns (non-link, non-index)
topic_columns = [c for c in raw_df.columns if not c.endswith('_links') and c != 'Unnamed: 0']
print(f'Topic columns: {topic_columns}')

# Build a long-format dataframe: (topic, abstract, title, link)
records = []
for topic in topic_columns:
    link_col = f'{topic}_links'
    for idx, row in raw_df.iterrows():
        abstract, title = parse_pubmed_entry(row[topic])
        if abstract is not None:
            records.append({
                'topic': topic,
                'abstract': abstract,
                'title': title,
                'link': row.get(link_col, None),
                'original_idx': idx
            })

pubmed_df = pd.DataFrame(records)
print(f'\nParsed PubMed DataFrame: {pubmed_df.shape}')
print(f'\nRecords per topic:')
print(pubmed_df['topic'].value_counts())
pubmed_df.head()

In [ ]:
# Load all BioASQ JSON files
bioasq_dir = Path('../data/raw/BioASQ')
bioasq_questions = []

for json_file in sorted(bioasq_dir.glob('*.json')):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, dict) and 'questions' in data:
        for q in data['questions']:
            q['source_file'] = json_file.name
            bioasq_questions.append(q)

bioasq_df = pd.DataFrame(bioasq_questions)
print(f'BioASQ DataFrame: {bioasq_df.shape}')
print(f'\nColumns: {bioasq_df.columns.tolist()}')
print(f'\nQuestion types:')
print(bioasq_df['type'].value_counts())
bioasq_df.head(3)

---
## 2. Missing Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: Heatmap of null values in raw dataframe (topic columns only)
null_matrix = raw_df[topic_columns].isnull()
sns.heatmap(null_matrix.T, cbar=True, yticklabels=True, cmap='YlOrRd',
            ax=axes[0], cbar_kws={'label': 'Missing (1=True)'})
axes[0].set_title('Missing Data Heatmap (Topic Columns)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Row Index')
axes[0].set_ylabel('')

# Right: Bar chart of non-null counts per topic
non_null_counts = raw_df[topic_columns].notna().sum().sort_values(ascending=True)
colors = sns.color_palette('viridis', len(non_null_counts))
axes[1].barh(non_null_counts.index, non_null_counts.values, color=colors)
axes[1].set_title('Non-Null Record Counts per Topic', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
for i, (val, name) in enumerate(zip(non_null_counts.values, non_null_counts.index)):
    axes[1].text(val + 100, i, f'{val:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/missing_data_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Missing Data Summary:')
print(f'Total rows: {len(raw_df):,}')
for col in topic_columns:
    null_pct = raw_df[col].isnull().mean() * 100
    print(f'  {col}: {raw_df[col].notna().sum():,} entries ({null_pct:.1f}% missing)')

---
## 3. Text Statistics

In [ ]:
# Compute text statistics
pubmed_df['abstract_word_count'] = pubmed_df['abstract'].apply(lambda x: len(str(x).split()))
pubmed_df['abstract_char_count'] = pubmed_df['abstract'].apply(lambda x: len(str(x)))
pubmed_df['title_word_count'] = pubmed_df['title'].apply(lambda x: len(str(x).split()))
pubmed_df['abstract_sent_count'] = pubmed_df['abstract'].apply(lambda x: len(sent_tokenize(str(x))))

print('📊 Overall Text Statistics:')
print(pubmed_df[['abstract_word_count', 'abstract_char_count', 'title_word_count', 'abstract_sent_count']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

metrics = [
    ('abstract_word_count', 'Abstract Word Count'),
    ('abstract_char_count', 'Abstract Character Count'),
    ('title_word_count', 'Title Word Count'),
    ('abstract_sent_count', 'Abstract Sentence Count')
]

for ax, (col, label) in zip(axes.flatten(), metrics):
    sns.boxplot(data=pubmed_df, x='topic', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{label} by Topic', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/text_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for topic in topic_columns:
    subset = pubmed_df[pubmed_df['topic'] == topic]
    axes[0].hist(subset['abstract_word_count'], bins=50, alpha=0.5, label=topic, density=True)
    axes[1].hist(subset['title_word_count'], bins=30, alpha=0.5, label=topic, density=True)

axes[0].set_title('Abstract Word Count Distribution by Topic', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8, loc='upper right')

axes[1].set_title('Title Word Count Distribution by Topic', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

---
## 4. Topic Distribution & Class Imbalance

Severe class imbalance can bias intent classification. This section quantifies the imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

topic_counts = pubmed_df['topic'].value_counts()

# Bar chart
colors = sns.color_palette('coolwarm', len(topic_counts))
bars = axes[0].bar(topic_counts.index, topic_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Sample Count per Topic', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, topic_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                f'{val:,}', ha='center', fontweight='bold', fontsize=10)

# Pie chart
axes[1].pie(topic_counts.values, labels=topic_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(topic_counts)), startangle=140,
            textprops={'fontsize': 10})
axes[1].set_title('Topic Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Imbalance metrics
max_count = topic_counts.max()
min_count = topic_counts.min()
print(f'\n⚖️ Class Imbalance Analysis:')
print(f'  Most represented: {topic_counts.idxmax()} ({max_count:,} samples)')
print(f'  Least represented: {topic_counts.idxmin()} ({min_count:,} samples)')
print(f'  Imbalance ratio: {max_count/min_count:.1f}:1')
print(f'\n  Recommendation: Use stratified sampling, SMOTE, or class weights for intent classification.')

---
## 5. Text Quality Audit

In [ ]:
# Exact duplicates
exact_dupes = pubmed_df.duplicated(subset=['abstract'], keep=False)
print(f'🔍 Duplicate Analysis:')
print(f'  Exact duplicate abstracts: {exact_dupes.sum():,} ({exact_dupes.mean()*100:.2f}%)')

# Cross-topic duplicates (same abstract appears in multiple topics)
abstract_topic_counts = pubmed_df.groupby('abstract')['topic'].nunique()
cross_topic = abstract_topic_counts[abstract_topic_counts > 1]
print(f'  Abstracts appearing in multiple topics: {len(cross_topic):,}')

if len(cross_topic) > 0:
    print(f'\n  Sample cross-topic duplicates:')
    for abs_text in cross_topic.head(3).index:
        topics = pubmed_df[pubmed_df['abstract'] == abs_text]['topic'].unique()
        print(f'    → Appears in: {list(topics)}')
        print(f'      Text preview: "{abs_text[:120]}..."')

In [ ]:
# Special character analysis
def analyze_text_quality(text):
    text = str(text)
    has_html = bool(re.search(r'<[^>]+>', text))
    has_urls = bool(re.search(r'http[s]?://\S+', text))
    has_emails = bool(re.search(r'\S+@\S+\.\S+', text))
    non_ascii = len(re.findall(r'[^\x00-\x7F]', text))
    is_very_short = len(text.split()) < 10
    return pd.Series({
        'has_html': has_html,
        'has_urls': has_urls, 
        'has_emails': has_emails,
        'non_ascii_chars': non_ascii,
        'is_very_short': is_very_short
    })

quality = pubmed_df['abstract'].apply(analyze_text_quality)

print('🧹 Text Quality Summary:')
print(f'  Entries with HTML tags: {quality["has_html"].sum():,}')
print(f'  Entries with URLs: {quality["has_urls"].sum():,}')
print(f'  Entries with emails: {quality["has_emails"].sum():,}')
print(f'  Entries with non-ASCII: {(quality["non_ascii_chars"] > 0).sum():,}')
print(f'  Very short abstracts (<10 words): {quality["is_very_short"].sum():,}')

# Show samples of problematic entries
if quality['is_very_short'].sum() > 0:
    short_mask = quality['is_very_short']
    print(f'\n  Sample short abstracts:')
    for _, row in pubmed_df[short_mask].head(3).iterrows():
        print(f'    [{row["topic"]}] "{row["abstract"][:100]}"')

In [ ]:
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42
    
    # Sample 500 abstracts for language detection (full corpus is slow)
    sample = pubmed_df.sample(min(500, len(pubmed_df)), random_state=42)
    sample_langs = sample['abstract'].apply(lambda x: detect(str(x)) if len(str(x)) > 20 else 'unknown')
    
    lang_counts = sample_langs.value_counts()
    print('🌐 Language Detection (sample of 500):')
    print(lang_counts)
    non_english = (sample_langs != 'en').sum()
    print(f'\n  Non-English entries in sample: {non_english} ({non_english/len(sample)*100:.1f}%)')
except ImportError:
    print('⚠️ langdetect not installed. Skipping language detection.')

---
## 6. NLP Feature Analysis

Linguistic features relevant to NER and intent classification.

In [ ]:
stop_words = list(stopwords.words('english'))

# TF-IDF per topic
print('📊 Top TF-IDF Terms per Topic (discriminative keywords):\n')

tfidf = TfidfVectorizer(max_features=5000, stop_words=stop_words, ngram_range=(1,2), max_df=0.8, min_df=5)
tfidf_matrix = tfidf.fit_transform(pubmed_df['abstract'])
feature_names = tfidf.get_feature_names_out()

for topic in topic_columns:
    mask = pubmed_df['topic'] == topic
    topic_tfidf = tfidf_matrix[mask].mean(axis=0).A1
    top_indices = topic_tfidf.argsort()[-15:][::-1]
    top_terms = [(feature_names[i], topic_tfidf[i]) for i in top_indices]
    print(f'  🏷️ {topic}:')
    print(f'    {" | ".join([f"{t[0]} ({t[1]:.3f})" for t in top_terms[:10]])}')
    print()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for ax, topic in zip(axes.flatten(), topic_columns):
    text = ' '.join(pubmed_df[pubmed_df['topic'] == topic]['abstract'].values)
    wc = WordCloud(width=600, height=400, background_color='white', 
                   max_words=100, colormap='viridis',
                   stopwords=set(stop_words),
                   collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(topic.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Clouds per Topic — Key Terms for Intent Classification', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/word_clouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        print('Downloading spaCy model...')
        spacy.cli.download('en_core_web_sm')
        nlp = spacy.load('en_core_web_sm')
    
    # Sample NER analysis on 200 abstracts
    ner_sample = pubmed_df.sample(min(200, len(pubmed_df)), random_state=42)
    
    entity_types = Counter()
    entity_examples = {}
    
    for _, row in ner_sample.iterrows():
        doc = nlp(row['abstract'][:1000])  # Limit text length for speed
        for ent in doc.ents:
            entity_types[ent.label_] += 1
            if ent.label_ not in entity_examples:
                entity_examples[ent.label_] = []
            if len(entity_examples[ent.label_]) < 3:
                entity_examples[ent.label_].append(ent.text)
    
    # Plot entity type distribution
    fig, ax = plt.subplots(figsize=(14, 6))
    ent_df = pd.DataFrame(entity_types.most_common(15), columns=['Entity Type', 'Count'])
    sns.barplot(data=ent_df, x='Entity Type', y='Count', palette='magma', ax=ax)
    ax.set_title('Named Entity Types Distribution (spaCy en_core_web_sm)', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('../data/processed/ner_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\n🏷️ Entity Type Examples:')
    for ent_type, count in entity_types.most_common(10):
        examples = entity_examples.get(ent_type, [])[:3]
        print(f'  {ent_type} ({count}): {examples}')
    
    print(f'\n💡 Insight: General-purpose NER (en_core_web_sm) may miss biomedical entities.')
    print(f'   Consider using scispaCy (en_core_sci_sm) or a biomedical NER model for better coverage.')

except Exception as e:
    print(f'⚠️ spaCy NER analysis skipped: {e}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for ax, topic in zip(axes.flatten(), topic_columns):
    topic_texts = pubmed_df[pubmed_df['topic'] == topic]['abstract']
    other_texts = pubmed_df[pubmed_df['topic'] != topic]['abstract']
    
    # Count bigrams
    cv = CountVectorizer(ngram_range=(2,2), stop_words=stop_words, max_features=2000)
    cv.fit(pubmed_df['abstract'])
    
    topic_counts = cv.transform(topic_texts).sum(axis=0).A1
    other_counts = cv.transform(other_texts).sum(axis=0).A1
    
    # Ratio of topic frequency to overall frequency (discriminative power)
    total = topic_counts + other_counts + 1
    ratio = topic_counts / total
    
    top_idx = ratio.argsort()[-10:][::-1]
    terms = cv.get_feature_names_out()
    
    ax.barh([terms[i] for i in top_idx], [ratio[i] for i in top_idx], color=sns.color_palette('rocket', 10))
    ax.set_title(topic.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Most Discriminative Bigrams per Topic — Intent Separators', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/discriminative_bigrams.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. BioASQ QA Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Question type distribution
type_counts = bioasq_df['type'].value_counts()
colors_qa = sns.color_palette('Set2', len(type_counts))
axes[0].bar(type_counts.index, type_counts.values, color=colors_qa, edgecolor='white')
axes[0].set_title('Question Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (t, v) in enumerate(zip(type_counts.index, type_counts.values)):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Question length distribution
bioasq_df['question_word_count'] = bioasq_df['body'].apply(lambda x: len(str(x).split()))
sns.boxplot(data=bioasq_df, x='type', y='question_word_count', palette='Set2', ax=axes[1])
axes[1].set_title('Question Length by Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Question Type')
axes[1].set_ylabel('Word Count')

# Snippets per question
bioasq_df['snippet_count'] = bioasq_df['snippets'].apply(lambda x: len(x) if isinstance(x, list) else 0)
sns.boxplot(data=bioasq_df, x='type', y='snippet_count', palette='Set2', ax=axes[2])
axes[2].set_title('Evidence Snippets per Question', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Question Type')
axes[2].set_ylabel('Snippet Count')

plt.tight_layout()
plt.savefig('../data/processed/bioasq_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 BioASQ Statistics:')
print(f'  Total questions: {len(bioasq_df):,}')
for qt in type_counts.index:
    print(f'  {qt}: {type_counts[qt]} ({type_counts[qt]/len(bioasq_df)*100:.1f}%)')
print(f'\n  Avg question length: {bioasq_df["question_word_count"].mean():.1f} words')
print(f'  Avg snippets per question: {bioasq_df["snippet_count"].mean():.1f}')

In [ ]:
# Analyze ideal answers
def get_answer_length(answer):
    if isinstance(answer, list):
        return np.mean([len(str(a).split()) for a in answer]) if answer else 0
    return len(str(answer).split())

bioasq_df['answer_word_count'] = bioasq_df['ideal_answer'].apply(get_answer_length)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=bioasq_df, x='answer_word_count', hue='type', bins=40, ax=axes[0], palette='Set2', alpha=0.7)
axes[0].set_title('Ideal Answer Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Word Count')

# Question keyword analysis
all_q_words = ' '.join(bioasq_df['body'].values)
wc = WordCloud(width=800, height=400, background_color='white', max_words=100,
               colormap='plasma', stopwords=set(stop_words)).generate(all_q_words)
axes[1].imshow(wc, interpolation='bilinear')
axes[1].set_title('BioASQ Question Keywords', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f'\n📝 Answer Statistics by Type:')
for qt in bioasq_df['type'].unique():
    subset = bioasq_df[bioasq_df['type'] == qt]
    print(f'  {qt}: avg answer length = {subset["answer_word_count"].mean():.1f} words')

---
## 8. Intent Classification Feasibility

Tests topic separability using TF-IDF features.

In [ ]:
# Subsample for t-SNE (max 500 per topic)
sampled_dfs = []
for topic in topic_columns:
    subset = pubmed_df[pubmed_df['topic'] == topic]
    sampled_dfs.append(subset.sample(min(500, len(subset)), random_state=42))
sampled = pd.concat(sampled_dfs, ignore_index=True)

tfidf_viz = TfidfVectorizer(max_features=3000, stop_words=stop_words)
X_viz = tfidf_viz.fit_transform(sampled['abstract'])

# t-SNE
print('Running t-SNE... (this may take a minute)')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_viz.toarray())

fig, ax = plt.subplots(figsize=(14, 10))
topics_unique = sampled['topic'].unique()
colors_tsne = sns.color_palette('husl', len(topics_unique))

for i, topic in enumerate(topics_unique):
    mask = sampled['topic'] == topic
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
              c=[colors_tsne[i]], label=topic.replace('_', ' ').title(),
              alpha=0.6, s=20, edgecolors='none')

ax.set_title('t-SNE Visualization of TF-IDF Features — Topic Clusters', fontsize=16, fontweight='bold')
ax.legend(fontsize=10, markerscale=3, loc='best')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('../data/processed/tsne_topics.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Interpretation: Well-separated clusters indicate topics are distinguishable by vocabulary alone.')
print('   Overlapping regions highlight topic pairs that may need more sophisticated features (e.g., NER, embeddings).')

In [ ]:
# Baseline intent classifier
print('🤖 Baseline Intent Classifier (Logistic Regression + TF-IDF)\n')

le = LabelEncoder()
y = le.fit_transform(pubmed_df['topic'])

tfidf_clf = TfidfVectorizer(max_features=5000, stop_words=stop_words, ngram_range=(1,2))
X = tfidf_clf.fit_transform(pubmed_df['abstract'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix — Baseline Intent Classifier', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify hardest pairs
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
print(f'\n⚠️ Most confused pair: {le.classes_[i]} ↔ {le.classes_[j]} ({cm[i,j]} misclassifications)')

---
## 9. Findings & Recommendations

In [ ]:
print('='*80)
print('📋 KEY FINDINGS & RECOMMENDATIONS FOR CONVERSATIONAL AI PIPELINE')
print('='*80)

print(f'''
🔹 DATA OVERVIEW
  • PubMed corpus: {len(pubmed_df):,} abstracts across {len(topic_columns)} biomedical topics
  • BioASQ QA: {len(bioasq_df):,} questions ({bioasq_df['type'].nunique()} types: factoid, yesno, list, summary)
  • Combined: rich resource for training a biomedical conversational AI

🔹 CLASS IMBALANCE (Critical for Intent Classification)
  • Ratio: {pubmed_df['topic'].value_counts().max() / pubmed_df['topic'].value_counts().min():.1f}:1 (largest to smallest topic)
  • Topics with <1000 samples: {', '.join([t for t, c in pubmed_df['topic'].value_counts().items() if c < 1000])}
  • ⚡ Action: Use class weights, SMOTE, or data augmentation

🔹 TEXT QUALITY
  • Most abstracts are well-formed scientific text
  • Minimal noise (HTML, URLs, non-English)
  • Average abstract length: ~{pubmed_df['abstract_word_count'].mean():.0f} words — sufficient for NER

🔹 NER & INTENT CLASSIFICATION READINESS
  • TF-IDF baseline achieves strong separation between most topics
  • General spaCy NER catches ORG, PERSON, GPE entities but misses biomedical terms
  • ⚡ Action: Use scispaCy or BioBERT for biomedical NER
  • ⚡ Action: Consider transformer-based embeddings (PubMedBERT) for better intent features

🔹 RECOMMENDED NEXT STEPS
  1. Data Preprocessing: Clean & deduplicate, standardize format
  2. NER Pipeline: Train/fine-tune biomedical NER model (scispaCy + custom entities)
  3. Intent Classifier: Fine-tune PubMedBERT or BioBERT on topic labels
  4. QA Integration: Use BioASQ for training the conversational QA component
  5. Evaluation: Cross-validation with stratified splits
''')

print('='*80)
print('✅ EDA Complete — Ready for NER & Intent Classification Pipeline')
print('='*80)